# Imports

In [2]:
import pandas as pd
from pathlib import Path

from pathlib import Path
import pandas as pd

In [3]:
import pyarrow 

# Code

## Change csv files

### Aqui eu mudei os csvs de cada subpasta, adicionando a coluna "folder" pra facilitar e trocando "category" por "vulnerability"

In [4]:
# Caminho base
base_path = Path("/home/luis/UFV/Datasets/clean_code")

# Procurar recursivamente arquivos CSV
csv_files = list(base_path.rglob("*.csv"))

if not csv_files:
    print(f"❌ Nenhum arquivo CSV encontrado em {base_path.resolve()}")
else:
    print(f"✅ {len(csv_files)} arquivos CSV encontrados em {base_path.resolve()}\n")

# Função auxiliar para mostrar um resumo de um DataFrame
def preview_df(df, lines=3):
    print(df.head(lines).to_string(index=False))
    print("...")

# Processamento principal
for i, csv_path in enumerate(csv_files, start=1):
    print(f"\n[{i}/{len(csv_files)}] Processando: {csv_path}")

    try:
        df = pd.read_csv(csv_path)

        # Verifica se a coluna 'category' existe
        if "category" not in df.columns:
            print("⚠️  Coluna 'category' não encontrada. Pulando arquivo.")
            continue

        print("📋 Colunas originais:", list(df.columns))

        # Criar coluna 'folder' com o conteúdo de 'category'
        df["folder"] = df["category"]

        # Renomear 'category' -> 'vulnerability'
        df = df.rename(columns={"category": "vulnerability"})

        print("✅ Colunas modificadas:", list(df.columns))

        # Mostrar pequena prévia
        print("\nPrévia dos dados modificados:")
        preview_df(df)

        # Sobrescrever o arquivo original
        df.to_csv(csv_path, index=False)
        print(f"💾 Arquivo sobrescrito com sucesso: {csv_path}")

    except Exception as e:
        print(f"❌ Erro ao processar {csv_path}: {e}")

print("\n🎉 Processamento concluído!")


✅ 4 arquivos CSV encontrados em /home/luis/UFV/Datasets/clean_code


[1/4] Processando: /home/luis/UFV/Datasets/clean_code/MANDO-HGT/MANDO-HGT.csv
📋 Colunas originais: ['name', 'line', 'category']
✅ Colunas modificadas: ['name', 'line', 'vulnerability', 'folder']

Prévia dos dados modificados:
                       name                                                                                                                                                                                                                                                                                                                                                   line  vulnerability         folder
access_control_buggy_19.sol                                                                                                    58-60, 69-72, 89-92, 98-102, 107-110, 116-119, 132-135, 144-147, 158-161, 167-170, 176-179, 185-187, 193-195, 201-204, 210-212, 218-221, 227-230, 236-239, 245-248, 254-256, 262-

### Aqui eu vou concatenar todos os datasets em um unico csv, pra facilitar minha vida depois

In [10]:
# === CONFIG ===
# aponte para a pasta que contém as subpastas dos repositórios (a que aparece na sua imagem)
CLEAN_CODE_ROOT = Path("/home/luis/UFV/Datasets/clean_code")  # ex: Path("/home/luis/smartcontract_benchmark/clean_code")
OUTPUT_CSV = CLEAN_CODE_ROOT / "all_contracts_with_code.csv"

# nome esperado do arquivo-índice em cada repositório
# (se os seus CSVs tiverem outros nomes, pode listar vários e o script usa o primeiro que existir)
CANDIDATE_CSV_NAMES = [
    "labels.csv",
    "dataset.csv",
    "contracts.csv",
    "index.csv",
    "vulns.csv",
    "vulnerabilities.csv"
]

def find_repo_csv(repo_dir: Path) -> Path | None:
    # tenta pelo(s) nome(s) comum(ns)
    for nm in CANDIDATE_CSV_NAMES:
        p = repo_dir / nm
        if p.exists():
            return p
    # fallback: pega o primeiro CSV encontrado no nível do repositório
    csvs = list(repo_dir.glob("*.csv"))
    if csvs:
        return csvs[0]
    return None

def ensure_sol_name(name: str) -> str:
    name = str(name).strip()
    return name if name.lower().endswith(".sol") else f"{name}.sol"

def read_code(path: Path) -> str | None:
    try:
        return path.read_text(encoding="utf-8", errors="replace")
    except Exception:
        return None

def locate_contract(repo_dir: Path, folder: str, name: str) -> Path | None:
    """
    1) tenta exatamente repo_dir / folder / name(.sol)
    2) se não achar, busca por nome em todo o repo (glob)
    """
    target_name = ensure_sol_name(name)
    # caminho direto usando 'folder' do CSV
    if folder and str(folder).strip() not in ("", ".", "/"):
        direct = (repo_dir / folder.strip("/")).expanduser()
        # às vezes folder já inclui o nome; tentamos as duas formas
        for p in [direct / target_name, direct]:
            if p.is_file() and p.suffix.lower() == ".sol":
                return p
    else:
        direct = repo_dir / target_name
        if direct.exists():
            return direct

    # fallback: busca pelo nome em qualquer subpasta
    matches = list(repo_dir.rglob(target_name))
    if matches:
        return matches[0]

    # último fallback: às vezes 'name' no CSV contém subcaminho
    name_path = repo_dir / name
    if name_path.exists() and name_path.suffix.lower() == ".sol":
        return name_path

    return None

def load_and_merge(root: Path) -> pd.DataFrame:
    all_rows = []
    repo_dirs = [p for p in root.iterdir() if p.is_dir()]
    print(f"🔍 Repositórios detectados em {root}: {[p.name for p in repo_dirs]}")

    for repo in repo_dirs:
        repo_csv = find_repo_csv(repo)
        if not repo_csv:
            print(f"⚠️  Nenhum CSV encontrado em {repo.name}; pulando.")
            continue

        print(f"📄 Lendo CSV: {repo_csv.relative_to(root)}")
        try:
            df = pd.read_csv(repo_csv, dtype={"name":"string","line":"string","vulnerability":"string","folder":"string"})
        except Exception as e:
            print(f"   ⚠️  Falha ao ler {repo_csv.name}: {e}")
            continue

        # normaliza colunas esperadas
        df.columns = [c.strip().lower() for c in df.columns]
        required = {"name","line","vulnerability","folder"}
        missing = required - set(df.columns)
        if missing:
            print(f"   ⚠️  CSV de {repo.name} sem colunas {missing}; pulando.")
            continue

        # adiciona colunas auxiliares
        df["repository"] = repo.name

        # resolve caminhos e lê código
        paths, codes = [], []
        for _, row in df.iterrows():
            p = locate_contract(repo, row.get("folder","") or "", row.get("name","") or "")
            paths.append(str(p) if p else None)
            codes.append(read_code(p) if p else None)

        df["path"] = paths
        df["code"] = codes

        # opcional: remove linhas sem código encontrado
        # df = df[df["code"].notna()]

        all_rows.append(df)

    if not all_rows:
        print("⛔ Nada para mesclar.")
        return pd.DataFrame(columns=["repository","folder","name","line","vulnerability","path","code"])

    merged = pd.concat(all_rows, ignore_index=True)

    # remove duplicatas evidentes (mesmo repo+path+linha+vuln)
    merged = merged.drop_duplicates(subset=["repository","path","line","vulnerability"], keep="first")

    # ordena pra ficar agradável
    merged = merged.sort_values(by=["repository","folder","name"], kind="stable").reset_index(drop=True)
    return merged

if __name__ == "__main__":
    df = load_and_merge(CLEAN_CODE_ROOT)
    print(f"\n✅ Linhas finais: {len(df)}")
    print(df[["repository","folder","name","line","vulnerability"]].head(10))
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"💾 Salvo em: {OUTPUT_CSV}")


🔍 Repositórios detectados em /home/luis/UFV/Datasets/clean_code: ['MANDO-HGT', 'HuangGai', 'DAppSCAN', 'smartbugs-curated']
📄 Lendo CSV: MANDO-HGT/labels.csv
📄 Lendo CSV: HuangGai/labels.csv
📄 Lendo CSV: DAppSCAN/labels.csv
📄 Lendo CSV: smartbugs-curated/labels.csv

✅ Linhas finais: 43483
  repository                               folder  \
0   DAppSCAN  SWC-100-Function Default Visibility   
1   DAppSCAN  SWC-100-Function Default Visibility   
2   DAppSCAN  SWC-100-Function Default Visibility   
3   DAppSCAN  SWC-100-Function Default Visibility   
4   DAppSCAN  SWC-100-Function Default Visibility   
5   DAppSCAN  SWC-100-Function Default Visibility   
6   DAppSCAN  SWC-100-Function Default Visibility   
7   DAppSCAN  SWC-100-Function Default Visibility   
8   DAppSCAN  SWC-100-Function Default Visibility   
9   DAppSCAN  SWC-100-Function Default Visibility   

                          name  line                        vulnerability  
0  AllocatedCrowdsaleMixin.sol   L38  SWC-100-Func

In [4]:
dataset_csv = pd.read_csv('/home/luis/UFV/Datasets/clean_code/all_contracts_with_code.csv')
dataset_csv.head()

,name,line,vulnerability,folder,repository,path,code
0,AllocatedCrowdsaleMixin.sol,L38,SWC-100-Function Default Visibility,SWC-100-Function Default Visibility,DAppSCAN,/home/luis/UFV/Datasets/clean_code/DAppSCAN/SW...,\n\n\n\n\n\npragma solidity ^0.4.8;\n\nimport ...
1,BMON_Z1.sol,L104,SWC-100-Function Default Visibility,SWC-100-Function Default Visibility,DAppSCAN,/home/luis/UFV/Datasets/clean_code/DAppSCAN/SW...,\n\n\n\npragma solidity >=0.7.0 <0.9.0;\n\n\ni...
2,BNRG.sol,L105,SWC-100-Function Default Visibility,SWC-100-Function Default Visibility,DAppSCAN,/home/luis/UFV/Datasets/clean_code/DAppSCAN/SW...,\n\n\n\npragma solidity >=0.7.0 <0.9.0;\n\n\ni...
3,BackedToken.sol,L28,SWC-100-Function Default Visibility,SWC-100-Function Default Visibility,DAppSCAN,/home/luis/UFV/Datasets/clean_code/DAppSCAN/SW...,"\npragma solidity ^0.5.0;\n\nimport ""@openzepp..."
4,Balancer.sol,L96,SWC-100-Function Default Visibility,SWC-100-Function Default Visibility,DAppSCAN,/home/luis/UFV/Datasets/clean_code/DAppSCAN/SW...,"\npragma solidity >=0.8.0 <0.9.0;\n\nimport ""@..."


In [5]:
dataset_csv.to_parquet("all_contracts_with_code.parquet")

ArrowKeyError: No type extension with name arrow.py_extension_type found